# Data Quality

Este notebook executa validações de qualidade sobre as principais tabelas Batch e Streaming do pipeline.

São avaliadas regras de completude, unicidade, domínio, relacionamento entre chaves e consistência dos indicadores.

# Configuração

São definidas as bibliotecas e a estrutura utilizada para registrar os resultados das validações.

### Bibliotecas

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

### Lista de Resultados

In [0]:
# Gera um identificador único para esta execução do processo de qualidade.
# Isso permite distinguir diferentes rodadas quando os resultados forem
# armazenados na tabela histórica quality.resultados_validacao.

execution_id = str(uuid.uuid4())

resultados = []

print("Execution ID:", execution_id)

### Função Registros

In [0]:
# Função central utilizada por todas as regras de qualidade.
# Ela padroniza o formato do resultado antes de adicioná-lo à lista `resultados`.
def registrar_resultado(
    camada, tabela, regra, descricao, qtd_registros, qtd_erros, criticidade="CRITICAL"
):

    # Define o status com base na quantidade de erros e na criticidade da regra.
    # - PASS: nenhum erro encontrado
    # - WARN: existem ocorrências, mas a regra foi classificada como alerta
    # - FAIL: existem ocorrências em uma regra crítica
    
    if qtd_erros == 0:
        status = "PASS"
    elif criticidade == "WARNING":
        status = "WARN"
    else:
        status = "FAIL"

    resultados.append(
        {
            "execution_id": execution_id,
            "camada": camada,
            "tabela": tabela,
            "regra": regra,
            "descricao": descricao,
            "criticidade": criticidade,
            "qtd_registros": int(qtd_registros),
            "qtd_erros": int(qtd_erros),
            "status": status,
            "validated_at": datetime.now(timezone.utc),
        }
    )

# Funções de Validação

As funções abaixo padronizam as principais regras de qualidade utilizadas no pipeline.

## Função de Valores obrigatórios

In [0]:
# Valida se colunas obrigatórias possuem valores nulos.
# Uma regra é registrada individualmente para cada coluna informada.
def validar_not_null(tabela, camada, colunas, criticidade="CRITICAL"):

    df = spark.table(tabela)
    total = df.count()

    for coluna in colunas:

        # Conta quantos registros possuem NULL na coluna avaliada.
        erros = df.filter(F.col(coluna).isNull()).count()

        registrar_resultado(
            camada=camada,
            tabela=tabela,
            regra="NOT_NULL",
            descricao=f"{coluna} não pode ser nulo",
            qtd_registros=total,
            qtd_erros=erros,
            criticidade=criticidade,
        )

## Função de Chaves Únicas

In [0]:
# Valida a unicidade de uma chave simples ou composta.
# A combinação das colunas recebidas deve identificar cada registro de forma única.

def validar_unicidade(tabela, camada, colunas, criticidade="CRITICAL"):

    df = spark.table(tabela)
    total = df.count()

    # Agrupa pela chave candidata e identifica combinações que aparecem mais de uma vez.
    # O cálculo soma apenas os registros excedentes de cada grupo duplicado.
    duplicados = (
        df.groupBy(*colunas)
        .count()
        .filter(F.col("count") > 1)
        .agg(F.sum(F.col("count") - 1).alias("qtd"))
        .collect()[0]["qtd"]
    )

    duplicados = duplicados or 0

    registrar_resultado(
        camada=camada,
        tabela=tabela,
        regra="UNIQUE",
        descricao=f"Chave única: {', '.join(colunas)}",
        qtd_registros=total,
        qtd_erros=duplicados,
        criticidade=criticidade,
    )

## Função de Intervalos Válidos

In [0]:
# Valida se os valores de uma coluna numérica estão dentro do intervalo esperado.
# Valores nulos são ignorados aqui, pois a completude é tratada pela regra NOT_NULL.
def validar_range(tabela, camada, coluna, minimo, maximo, criticidade="CRITICAL"):

    df = spark.table(tabela)
    total = df.count()

    # Considera erro apenas valores não nulos abaixo do mínimo ou acima do máximo.
    erros = df.filter(
        F.col(coluna).isNotNull()
        & ((F.col(coluna) < minimo) | (F.col(coluna) > maximo))
    ).count()

    registrar_resultado(
        camada=camada,
        tabela=tabela,
        regra="RANGE",
        descricao=f"{coluna} deve estar entre {minimo} e {maximo}",
        qtd_registros=total,
        qtd_erros=erros,
        criticidade=criticidade,
    )

## Função de Regras de Consistência

In [0]:
# Função genérica para validar regras de consistência entre colunas.
# A expressão `condicao_invalida` deve representar exatamente os registros incorretos.
def validar_condicao(
    tabela, camada, nome_regra, descricao, condicao_invalida, criticidade="CRITICAL"
):

    df = spark.table(tabela)
    total = df.count()

    erros = df.filter(condicao_invalida).count()

    # Conta quantos registros violam a condição de negócio definida.
    registrar_resultado(
        camada=camada,
        tabela=tabela,
        regra=nome_regra,
        descricao=descricao,
        qtd_registros=total,
        qtd_erros=erros,
        criticidade=criticidade,
    )

# Validações Camada Silver
Na Silver são avaliadas principalmente completude, unicidade das chaves e domínio dos indicadores.

## Tabela de Alunos

In [0]:
# -----------------------------
# Validações da tabela Silver de alunos
# -----------------------------

# Campos que compõem a identificação e o contexto básico do aluno devem estar preenchidos.
validar_not_null(
    tabela="silver.alunos",
    camada="SILVER",
    colunas=["ano", "id_aluno", "id_municipio", "rede"],
)

# A combinação ano + id_aluno foi definida como chave candidata da tabela.
validar_unicidade(tabela="silver.alunos", camada="SILVER", colunas=["ano", "id_aluno"])

# O indicador alfabetizado é binário, portanto somente 0 e 1 são valores válidos.
validar_range(tabela="silver.alunos", camada="SILVER", coluna="alfabetizado", minimo=0, maximo=1)

## Tabela de Metas

**Tratamento de metas ausentes**

Foram identificados registros sem `meta_alfabetizacao` em combinações específicas de território e período. Como a ausência está presente na fonte e não representa erro de transformação, a regra é classificada como `WARNING`.

Os registros são preservados e não recebem preenchimento artificial.

In [0]:
# -----------------------------
# Validações da tabela Silver de metas
# -----------------------------

# Campos estruturais necessários para identificar cada meta devem estar preenchidos.
validar_not_null(
    tabela="silver.metas",
    camada="SILVER",
    colunas=[
        "ano_referencia",
        "ano_meta",
        "nivel_geografico",
        "id_geografia",
        "rede"
    ]
)

# A ausência de meta ocorre em combinações específicas da própria fonte.
# Por isso, o caso é preservado e tratado como WARNING em vez de FAIL.
validar_not_null(
    tabela="silver.metas",
    camada="SILVER",
    colunas=[
        "meta_alfabetizacao"
    ],
    criticidade="WARNING"
)

# Cada combinação de versão da meta, território, rede e ano-meta deve ser única.
validar_unicidade(
    tabela="silver.metas",
    camada="SILVER",
    colunas=["ano_referencia", "ano_meta", "nivel_geografico", "id_geografia", "rede"],
)

# Quando informada, a meta de alfabetização deve representar um percentual entre 0 e 100.
validar_range(
    tabela="silver.metas",
    camada="SILVER",
    coluna="meta_alfabetizacao",
    minimo=0,
    maximo=100,
)

# Relacionamento entre Tabelas
É validado se os registros da distribuição de proficiência possuem território correspondente na tabela consolidada de resultados territoriais.

In [0]:
# -----------------------------
# Integridade referencial entre tabelas territoriais da Silver
# -----------------------------

# Seleciona apenas as chaves territoriais existentes na distribuição por nível.
df_distribuicao = (
    spark.table("silver.distribuicao_niveis")
    .select("ano", "nivel_geografico", "id_geografia", "rede")
    .distinct()
)

# Seleciona as mesmas chaves na tabela consolidada de resultados territoriais.
df_territorial = (
    spark.table("silver.resultados_territoriais")
    .select("ano", "nivel_geografico", "id_geografia", "rede")
    .distinct()
)


# O left_anti retorna apenas registros da distribuição que NÃO possuem
# uma chave correspondente em resultados_territoriais.
df_sem_correspondencia = df_distribuicao.join(
    df_territorial,
    on=["ano", "nivel_geografico", "id_geografia", "rede"],
    how="left_anti",
)

qtd_erros = df_sem_correspondencia.count()
qtd_total = df_distribuicao.count()

# Registra a quantidade de territórios sem correspondência como erro de integridade.
registrar_resultado(
    camada="SILVER",
    tabela="silver.distribuicao_niveis",
    regra="REFERENTIAL_INTEGRITY",
    descricao="Territórios devem existir em silver.resultados_territoriais",
    qtd_registros=qtd_total,
    qtd_erros=qtd_erros,
)

# Validações Camada Gold
Na Gold são verificadas regras de domínio e consistência entre os indicadores calculados.

## Indicadores por Rede

In [0]:
# -----------------------------
# Validações da Gold de indicadores por rede
# -----------------------------

# Todos os indicadores percentuais calculados devem permanecer entre 0% e 100%.
for coluna in [
    "taxa_presenca_pct",
    "taxa_preenchimento_pct",
    "taxa_preenchimento_presentes_pct",
    "taxa_alfabetizacao_avaliados_pct",
    "taxa_alfabetizacao_ponderada_pct",
]:

    validar_range(
        tabela="gold.indicadores_rede",
        camada="GOLD",
        coluna=coluna,
        minimo=0,
        maximo=100,
    )

# A quantidade de alunos presentes nunca pode superar o total de alunos.
validar_condicao(
    tabela="gold.indicadores_rede",
    camada="GOLD",
    nome_regra="CONSISTENCY",
    descricao="Alunos presentes não podem superar o total de alunos",
    condicao_invalida=(
        F.col("alunos_presentes")
        > F.col("total_alunos")
    )
)

# Provas preenchidas pressupõem presença, portanto não podem superar alunos presentes.
validar_condicao(
    tabela="gold.indicadores_rede",
    camada="GOLD",
    nome_regra="CONSISTENCY",
    descricao="Provas preenchidas não podem superar alunos presentes",
    condicao_invalida=(
        F.col("provas_preenchidas")
        > F.col("alunos_presentes")
    )
)

# Um aluno só pode ser classificado como alfabetizado se possuir avaliação preenchida.
validar_condicao(
    tabela="gold.indicadores_rede",
    camada="GOLD",
    nome_regra="CONSISTENCY",
    descricao="Alunos alfabetizados não podem superar provas preenchidas",
    condicao_invalida=(
        F.col("alunos_alfabetizados")
        > F.col("provas_preenchidas")
    )
)

## Acompanhamento das Metas

In [0]:
# -----------------------------
# Validações da Gold de acompanhamento de metas
# -----------------------------

# Taxa observada e meta são percentuais e, quando preenchidas, devem estar entre 0 e 100.
validar_range(
    tabela="gold.acompanhamento_metas",
    camada="GOLD",
    coluna="taxa_alfabetizacao",
    minimo=0,
    maximo=100,
)

validar_range(
    tabela="gold.acompanhamento_metas",
    camada="GOLD",
    coluna="meta_alfabetizacao",
    minimo=0,
    maximo=100,
)

# Confere se o gap armazenado corresponde à fórmula:
# taxa observada - meta de alfabetização.
# É utilizada tolerância de 0,01 para evitar diferenças irrelevantes de arredondamento.
validar_condicao(
    tabela="gold.acompanhamento_metas",
    camada="GOLD",
    nome_regra="CONSISTENCY_GAP",
    descricao="Gap deve corresponder à diferença entre taxa e meta",
    condicao_invalida=(
        F.col("taxa_alfabetizacao").isNotNull()
        & F.col("meta_alfabetizacao").isNotNull()
        & (
            F.abs(
                F.col("gap_meta_pp")
                - (F.col("taxa_alfabetizacao") - F.col("meta_alfabetizacao"))
            )
            > 0.01
        )
    ),
)

# Verifica se a flag de atingimento está coerente com a comparação entre taxa e meta:
# 1 quando taxa >= meta e 0 quando taxa < meta.
validar_condicao(
    tabela="gold.acompanhamento_metas",
    camada="GOLD",
    nome_regra="CONSISTENCY_FLAG",
    descricao="Flag de atingimento deve estar coerente com taxa e meta",
    condicao_invalida=(
        F.col("taxa_alfabetizacao").isNotNull()
        & F.col("meta_alfabetizacao").isNotNull()
        & (
            (
                (F.col("taxa_alfabetizacao") >= F.col("meta_alfabetizacao"))
                & (F.col("flag_atingiu_meta") != 1)
            )
            | (
                (F.col("taxa_alfabetizacao") < F.col("meta_alfabetizacao"))
                & (F.col("flag_atingiu_meta") != 0)
            )
        )
    ),
)

## Resumo de Atingimento

In [0]:
# -----------------------------
# Validações da Gold de resumo de atingimento
# -----------------------------

# O número de regiões com resultado avaliável não pode ser maior que o universo total.
validar_condicao(
    tabela="gold.resumo_atingimento",
    camada="GOLD",
    nome_regra="CONSISTENCY",
    descricao="Regiões avaliáveis não podem superar o total de regiões",
    condicao_invalida=(F.col("regioes_avaliaveis") > F.col("total_regioes")),
)

# Toda região avaliável deve estar classificada em uma das duas categorias:
# atingiu a meta ou não atingiu a meta.
validar_condicao(
    tabela="gold.resumo_atingimento",
    camada="GOLD",
    nome_regra="CONSISTENCY",
    descricao="Atingiram + não atingiram deve ser igual às regiões avaliáveis",
    condicao_invalida=(
        (F.col("regioes_que_atingiram") + F.col("regioes_que_nao_atingiram"))
        != F.col("regioes_avaliaveis")
    ),
)

# Validação Streaming
São verificadas a unicidade dos eventos e a preservação dos registros entre Silver e Gold Streaming.

In [0]:
# -----------------------------
# Validações do pipeline Streaming
# -----------------------------

# Cada evento deve possuir event_id único na Silver Streaming.
validar_unicidade(
    tabela="silver.eventos_alfabetizacao_streaming",
    camada="SILVER_STREAMING",
    colunas=["event_id"],
)

# O mesmo princípio de unicidade deve ser preservado na Gold Streaming.
validar_unicidade(
    tabela="gold.acompanhamento_metas_streaming",
    camada="GOLD_STREAMING",
    colunas=["event_id"],
)

# Seleciona os identificadores dos eventos processados em cada camada.
df_stream_silver = spark.table("silver.eventos_alfabetizacao_streaming").select(
    "event_id"
)

df_stream_gold = spark.table("gold.acompanhamento_metas_streaming").select("event_id")

# O left_anti identifica eventos que chegaram à Silver, mas não foram encontrados na Gold.
qtd_sem_gold = df_stream_silver.join(
    df_stream_gold, on="event_id", how="left_anti"
).count()

registrar_resultado(
    camada="STREAMING",
    tabela="gold.acompanhamento_metas_streaming",
    regra="REFERENTIAL_INTEGRITY",
    descricao="Todo evento Silver deve possuir correspondente na Gold",
    qtd_registros=df_stream_silver.count(),
    qtd_erros=qtd_sem_gold,
)

# Resultado das Validações
Os resultados são consolidados em uma tabela de auditoria, permitindo identificar regras aprovadas e eventuais falhas de qualidade.

Criando o schema

In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS quality"
)

In [0]:
# Criando o dataframe
df_resultados = spark.createDataFrame(resultados)


# Persistindo em formato delta
(
    df_resultados.write.format("delta")
    .mode("append")
    .saveAsTable("quality.resultados_validacao")
)

Visualizações

In [0]:
%sql
-- Resumo dos resultados históricos por status.
-- Permite visualizar rapidamente quantas regras passaram, geraram alerta ou falharam.
select status, count(*) from quality.resultados_validacao group by 1

In [0]:
%sql
-- Consulta detalhada do histórico de validações armazenado na camada de qualidade.
select * from quality.resultados_validacao